# M3_05: Pattern Analysis - Commuter vs Tourist

**Purpose**: Discover the two distinct rental patterns in OV-fiets data to inform your track selection

**Learning Objectives**:
- Identify commuter patterns (weekday peaks, short rentals)
- Identify tourist patterns (weekend/holiday spikes, long rentals)
- Visualize temporal differences between user types
- Make an informed decision about Track A vs Track B

**Duration**: 45-60 minutes

---

## 🎯 Decision Support Context

This notebook is **critical for track selection**. You'll analyze:

1. **Commuter Use Case** → Suggests Track A (Classification)
   - Problem: "Will a bike be available at 8:30 AM?"
   - Prediction: Binary (yes/no)
   - Horizon: 2-4 hours ahead

2. **Tourist Use Case** → Suggests Track B (Regression/Time Series)
   - Problem: "How many bikes will be available this Saturday?"
   - Prediction: Count (0-100+)
   - Horizon: 1-3 days ahead

After this analysis, review:
- [Use Case Comparison](../../docs/guides/use_case_comparison.md)
- [Learning Pathways](../../docs/guides/learning_pathways.md)

---

## Part 1: Setup & Data Loading

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set visualization defaults
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✅ Imports complete")

In [ ]:
# Load merged dataset from Module 2
# TODO: Update path to your merged bike + weather data
df = pd.read_csv('../../data/processed/bike_weather_merged.csv', parse_dates=['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## Part 2: Temporal Feature Engineering

Create features to help identify patterns

In [ ]:
# Extract temporal features
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['day_name'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month
df['date'] = df['timestamp'].dt.date

# Commuter peak hours (morning and evening rush)
df['is_peak_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

print("✅ Temporal features created")
df[['timestamp', 'hour', 'day_of_week', 'day_name', 'is_weekend', 'is_peak_hour']].head()

## Part 3: Pattern 1 - Weekday vs Weekend Analysis

**Hypothesis**: Commuters dominate weekdays, tourists dominate weekends

In [ ]:
# Compare weekday vs weekend bike availability patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Weekday pattern
weekday_data = df[df['is_weekend'] == 0].groupby('hour')['bikes_available'].mean()
axes[0].plot(weekday_data.index, weekday_data.values, marker='o', linewidth=2, color='#2E86AB')
axes[0].axvspan(7, 9, alpha=0.2, color='red', label='Morning Peak')
axes[0].axvspan(17, 19, alpha=0.2, color='orange', label='Evening Peak')
axes[0].set_title('Weekday Pattern: Commuter Peaks', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Bikes Available')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Weekend pattern
weekend_data = df[df['is_weekend'] == 1].groupby('hour')['bikes_available'].mean()
axes[1].plot(weekend_data.index, weekend_data.values, marker='o', linewidth=2, color='#A23B72')
axes[1].set_title('Weekend Pattern: Tourist Gradual Rise', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Average Bikes Available')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("- Weekdays show clear peaks at 8-9 AM and 5-6 PM (commuter rush)")
print("- Weekends show gradual decline throughout the day (tourist usage)")

## Part 4: Pattern 2 - Peak Hour Analysis

**Hypothesis**: Peak hour demand is predictable and short-term

In [ ]:
# Heatmap: Bike availability by day of week and hour
pivot_data = df.groupby(['day_name', 'hour'])['bikes_available'].mean().reset_index()
pivot_table = pivot_data.pivot(index='day_name', columns='hour', values='bikes_available')

# Reorder days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot_table = pivot_table.reindex(day_order)

plt.figure(figsize=(16, 6))
sns.heatmap(pivot_table, cmap='RdYlGn', annot=False, fmt='.1f', cbar_kws={'label': 'Avg Bikes Available'})
plt.title('Bike Availability Heatmap: Day × Hour', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("- Clear depletion (red) during weekday mornings/evenings")
print("- Weekend patterns are more gradual and consistent")

## Part 5: Pattern 3 - Holiday vs Non-Holiday

**Hypothesis**: Holidays behave like extended weekends (tourist pattern)

In [ ]:
# TODO: Add Dutch national holidays
# You can use pandas holiday calendar or manually define dates
# Example: Christmas, New Year, King's Day, etc.

# For now, let's analyze December as a proxy for holiday season
df['is_december'] = (df['month'] == 12).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Regular months
regular_data = df[df['is_december'] == 0].groupby('hour')['bikes_available'].mean()
axes[0].plot(regular_data.index, regular_data.values, marker='o', linewidth=2)
axes[0].set_title('Regular Months Pattern', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Bikes Available')
axes[0].grid(True, alpha=0.3)

# December (holiday season)
december_data = df[df['is_december'] == 1].groupby('hour')['bikes_available'].mean()
axes[1].plot(december_data.index, december_data.values, marker='o', linewidth=2, color='#C73E1D')
axes[1].set_title('December Pattern (Holiday Season)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Average Bikes Available')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("- December shows increased tourist-like behavior")
print("- Holiday periods require different forecasting approach")

## Part 6: Statistical Summary

Quantify the differences between patterns

In [ ]:
# Compare statistics: Weekday vs Weekend
summary = pd.DataFrame({
    'Metric': ['Mean Availability', 'Std Dev', 'Peak Hour Mean', 'Off-Peak Mean', 'Volatility (CV)'],
    'Weekday': [
        df[df['is_weekend'] == 0]['bikes_available'].mean(),
        df[df['is_weekend'] == 0]['bikes_available'].std(),
        df[(df['is_weekend'] == 0) & (df['is_peak_hour'] == 1)]['bikes_available'].mean(),
        df[(df['is_weekend'] == 0) & (df['is_peak_hour'] == 0)]['bikes_available'].mean(),
        df[df['is_weekend'] == 0]['bikes_available'].std() / df[df['is_weekend'] == 0]['bikes_available'].mean()
    ],
    'Weekend': [
        df[df['is_weekend'] == 1]['bikes_available'].mean(),
        df[df['is_weekend'] == 1]['bikes_available'].std(),
        df[(df['is_weekend'] == 1) & (df['is_peak_hour'] == 1)]['bikes_available'].mean(),
        df[(df['is_weekend'] == 1) & (df['is_peak_hour'] == 0)]['bikes_available'].mean(),
        df[df['is_weekend'] == 1]['bikes_available'].std() / df[df['is_weekend'] == 1]['bikes_available'].mean()
    ]
})

summary['Difference %'] = ((summary['Weekend'] - summary['Weekday']) / summary['Weekday'] * 100).round(2)

print("\n📊 Statistical Comparison: Weekday vs Weekend\n")
print(summary.to_string(index=False))

print("\n💡 Key Insights:")
print(f"- Weekday peak hours have {summary.loc[2, 'Difference %']:.1f}% different availability than weekends")
print(f"- Weekend patterns are {'more' if summary.loc[4, 'Weekend'] > summary.loc[4, 'Weekday'] else 'less'} volatile (CV = {summary.loc[4, 'Weekend']:.2f})")

## Part 7: Track Decision Support Visualization

In [ ]:
# Create a comprehensive comparison plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Hourly patterns by day type
df.groupby(['is_weekend', 'hour'])['bikes_available'].mean().unstack(0).plot(ax=axes[0,0], marker='o')
axes[0,0].set_title('Track A: Short-term Prediction (2-4 hours)', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Average Bikes Available')
axes[0,0].legend(['Weekday (Commuter)', 'Weekend (Tourist)'])
axes[0,0].axvspan(7, 9, alpha=0.1, color='red')
axes[0,0].axvspan(17, 19, alpha=0.1, color='orange')
axes[0,0].grid(True, alpha=0.3)

# 2. Daily aggregates over time
daily_avg = df.groupby('date')['bikes_available'].mean()
daily_avg.plot(ax=axes[0,1], color='purple', linewidth=1.5)
axes[0,1].set_title('Track B: Long-term Forecasting (1-3 days)', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Date')
axes[0,1].set_ylabel('Daily Average Bikes')
axes[0,1].grid(True, alpha=0.3)

# 3. Distribution comparison
df.boxplot(column='bikes_available', by='is_weekend', ax=axes[1,0])
axes[1,0].set_title('Distribution: Weekday vs Weekend', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Day Type (0=Weekday, 1=Weekend)')
axes[1,0].set_ylabel('Bikes Available')
plt.sca(axes[1,0])
plt.xticks([1, 2], ['Weekday\n(Commuter)', 'Weekend\n(Tourist)'])

# 4. Prediction horizon comparison
horizons = ['2-4 hours\n(Track A)', '1-3 days\n(Track B)']
complexity = [3, 8]  # Relative complexity scores
axes[1,1].bar(horizons, complexity, color=['#2E86AB', '#A23B72'])
axes[1,1].set_title('Prediction Complexity', fontsize=12, fontweight='bold')
axes[1,1].set_ylabel('Relative Complexity (1-10)')
axes[1,1].set_ylim(0, 10)
for i, v in enumerate(complexity):
    axes[1,1].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Pattern analysis complete!")

## Part 8: Your Decision - Summary & Next Steps

### What You've Discovered

Through this analysis, you've identified:

**1. Commuter Pattern (Track A Focus)**
- ✅ Clear weekday peaks at 8-9 AM and 5-6 PM
- ✅ Short prediction horizon (2-4 hours)
- ✅ Binary problem: "Is a bike available?"
- ✅ Predictable patterns

**2. Tourist Pattern (Track B Focus)**
- ✅ Weekend and holiday demand spikes
- ✅ Longer prediction horizon (1-3 days)
- ✅ Regression problem: "How many bikes available?"
- ✅ Seasonal and event-driven

---

### 🎯 Track Selection Guide

| Criteria | Track A | Track B |
|----------|---------|----------|
| **Problem Type** | Classification | Regression + Time Series |
| **Difficulty** | Beginner | Advanced |
| **Duration** | 20-30 hrs | 30-45 hrs |
| **Prerequisites** | Python basics | Python + ML fundamentals |
| **Use Case** | Commuter (short-term) | Tourist (multi-day) |
| **Models** | 3 classifiers | 6+ models (regression + TS) |

---

### 📚 Next Steps

**Before choosing, review these guides:**

1. **[Use Case Comparison](../../docs/guides/use_case_comparison.md)**
   - Detailed decision tree
   - Prerequisites checklist
   - Feature engineering comparison

2. **[Learning Pathways Guide](../../docs/guides/learning_pathways.md)**
   - Complete module sequences
   - Self-assessment quiz
   - Learning strategies

3. **[Course Structure](../../docs/guides/course_structure_dual_track.md)**
   - Full module breakdown
   - Track divergence/convergence

---

### ✅ Make Your Choice

**Track A (Classification):**
```python
# Proceed to Module 04, follow track_a_commuter/ folders
```

**Track B (Both Tracks):**
```python
# Proceed to Module 04, complete BOTH track folders
```

**Still Unsure:**
```python
# Start with Track A, you can always return for Track B!
```

---

### 💬 Questions?

Review the [Module 03 README](README.md) for the complete decision point section.

**Good luck with your chosen track!** 🚀